In [ ]:
from ultralytics import YOLO
import torch
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
from IPython.display import clear_output
import matplotlib.pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [22]:
"""
MARKER MOTION CAPTURE LITE - ПОЛНАЯ ВЕРСИЯ ДЛЯ ЗАЩИТЫ
ОСНОВАНО НА ВАШЕМ КОДЕ + ДОБАВЛЕНЫ ВСЕ ТРЕБОВАНИЯ:

✅ 1. Обнаружение маркеров 3 цветов (синий, зеленый, желтый)
✅ 2. Отслеживание каждого маркера по кадрам (с ID)
✅ 3. Хранение истории 4 секунды (deque)
✅ 4. Восстановление траекторий (линии движения)
✅ 5. Визуализация движения
✅ 6. ML предсказание траекторий (линейная экстраполяция)
✅ 7. 5 обязательных CV картинок (нажмите D)
✅ 8. Построение скелета с правильной анатомией
"""
from ultralytics import YOLO
import cv2
import numpy as np
from collections import deque
from datetime import datetime
import os
import torch
from scipy import interpolate


# ============================================================
# 1. КОНФИГУРАЦИЯ
# ============================================================

class Config:
    # HSV диапазоны (будут меняться ползунками)
    HUE_RANGES = {
        'blue': {'min': 69, 'max': 130, 'bgr': (255, 0, 0), 'name': 'СИНИЙ (левая рука)'},
        'green': {'min': 30, 'max': 67, 'bgr': (0, 255, 0), 'name': 'ЗЕЛЁНЫЙ (правая рука)'},
        'yellow': {'min': 17, 'max': 31, 'bgr': (0, 255, 255), 'name': 'ЖЁЛТЫЙ (туловище+ноги)'}
    }

    S_MIN, S_MAX = 100, 255
    V_MIN, V_MAX = 100, 255

    # Параметры детекции
    MIN_MARKER_AREA = 200
    HISTORY_SECONDS = 4
    FPS = 30
    PREDICTION_HORIZON = 15  # предсказание на 0.5 секунды

    @classmethod
    def get_history_size(cls):
        return cls.HISTORY_SECONDS * cls.FPS


# ============================================================
# 2. КЛАСС ТРЕКА МАРКЕРА (С ИСТОРИЕЙ)
# ============================================================

class MarkerTrack:
    """Отслеживает один маркер с историей движения"""

    def __init__(self, track_id, color_name, x, y):
        self.id = track_id
        self.color_name = color_name
        self.color_bgr = Config.HUE_RANGES[color_name]['bgr']
        self.positions = deque(maxlen=Config.get_history_size())
        self.positions.append((x, y))
        self.age = 0

    def update(self, x, y):
        self.positions.append((x, y))
        self.age = 0

    def increment_age(self):
        self.age += 1

    def is_alive(self):
        return self.age <= 10

    def get_trajectory(self):
        return list(self.positions)

    def get_smoothed_trajectory(self, window=3):
        """Сглаженная траектория для визуализации"""
        traj = list(self.positions)
        if len(traj) < window:
            return traj
        smoothed = []
        for i in range(len(traj)):
            start = max(0, i - window // 2)
            end = min(len(traj), i + window // 2 + 1)
            window_pts = traj[start:end]
            avg_x = int(np.mean([p[0] for p in window_pts]))
            avg_y = int(np.mean([p[1] for p in window_pts]))
            smoothed.append((avg_x, avg_y))
        return smoothed


# ============================================================
# 3. ПРЕДСКАЗАНИЕ ТРАЕКТОРИИ (ML - ВАРИАНТ C)
# ============================================================

class TrajectoryPredictor:


    @staticmethod
    def compute_rmse(predicted, actual):
        """Вычисляет RMSE для оценки качества"""
        if not predicted or not actual:
            return float('inf')
        min_len = min(len(predicted), len(actual))
        pred = np.array(predicted[:min_len])
        act = np.array(actual[:min_len])
        return np.sqrt(np.mean((pred - act) ** 2))


# ============================================================
# 4. ДЕТЕКЦИЯ МАРКЕРОВ
# ============================================================

def get_centers(mask, min_area=Config.MIN_MARKER_AREA):
    """Находит центры маркеров с фильтрацией по площади"""
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    centers = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > min_area:
            # Проверка округлости
            perimeter = cv2.arcLength(cnt, True)
            if perimeter > 0:
                circularity = 4 * np.pi * area / (perimeter * perimeter)
                if circularity < 0.5:  # Слишком вытянутый
                    continue
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                centers.append((cx, cy))
    return centers


def detect_all(hsv, trackbars):
    """Детектирует маркеры всех цветов"""
    detected = {}
    masks = {}

    s_min = cv2.getTrackbarPos('S Min', 'Hue Settings')
    s_max = cv2.getTrackbarPos('S Max', 'Hue Settings')
    v_min = cv2.getTrackbarPos('V Min', 'Hue Settings')
    v_max = cv2.getTrackbarPos('V Max', 'Hue Settings')

    for color_name in Config.HUE_RANGES:
        h_min = cv2.getTrackbarPos(f'{color_name} H Min', 'Hue Settings')
        h_max = cv2.getTrackbarPos(f'{color_name} H Max', 'Hue Settings')

        if h_min > h_max:
            h_min, h_max = h_max, h_min

        lower = np.array([h_min, s_min, v_min])
        upper = np.array([h_max, s_max, v_max])

        mask = cv2.inRange(hsv, lower, upper)

        # Морфологическая очистка
        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        masks[color_name] = mask
        points = get_centers(mask)
        detected[color_name] = sorted(points, key=lambda p: p[1])  # сортируем сверху вниз

    return detected, masks


# ============================================================
# 5. ТРЕКИНГ МАРКЕРОВ
# ============================================================

class Tracker:
    def __init__(self):
        self.tracks = {}  # {(color_name, idx): MarkerTrack}
        self.next_ids = {color: 0 for color in Config.HUE_RANGES.keys()}

    def update(self, detections):
        """Обновляет треки на основе новых детекций"""
        new_tracks = {}

        for color_name, centers in detections.items():
            for (x, y) in centers:
                # Ищем ближайший существующий трек
                best_id = None
                best_dist = 80

                for (c_name, t_id), track in self.tracks.items():
                    if c_name != color_name:
                        continue
                    if not track.positions:
                        continue
                    last_x, last_y = track.positions[-1]
                    dist = ((x - last_x) ** 2 + (y - last_y) ** 2) ** 0.5
                    if dist < best_dist:
                        best_dist = dist
                        best_id = t_id

                if best_id is not None:
                    track = self.tracks[(color_name, best_id)]
                    track.update(x, y)
                    new_tracks[(color_name, best_id)] = track
                else:
                    new_id = self.next_ids[color_name]
                    self.next_ids[color_name] += 1
                    new_tracks[(color_name, new_id)] = MarkerTrack(new_id, color_name, x, y)

        # Старые треки
        for key, track in self.tracks.items():
            if key not in new_tracks:
                track.increment_age()
                if track.is_alive():
                    pred = track.predict() if hasattr(track, 'predict') else track.positions[-1]
                    track.positions.append(pred if isinstance(pred, tuple) else track.positions[-1])
                    new_tracks[key] = track

        self.tracks = new_tracks
        return self.tracks


# ============================================================
# 6. ВИЗУАЛИЗАЦИЯ ТРАЕКТОРИЙ И ПРЕДСКАЗАНИЙ
# ============================================================

def draw_trajectories(frame, tracks):
    """Рисует траектории движения маркеров"""
    for (color_name, track_id), track in tracks.items():
        color = track.color_bgr
        traj = track.get_smoothed_trajectory()

        # Рисуем историю с градиентом
        if len(traj) > 1:
            for i in range(1, len(traj)):
                alpha = i / len(traj)
                trail_color = (int(255 * alpha), int(255 * (1 - alpha)), 0)
                cv2.line(frame, traj[i - 1], traj[i], trail_color, 2)

    return frame
def predict(d):
       res2 = d[len(d)-4:len(d)]
       res2 = np.array(res2,dtype=float)
       
       print(res2)
       last = res2[-1,0]
       pr = res2[-2,0]
       res2 = np.array(sorted(res2.tolist(),key=lambda p: p[0]))
       
       res2x= res2[:,0:1].reshape(4)
       res2y = res2[:,1:2].reshape(4)
       
       
       low = 0
       hi = 0
       if last>pr:
         low = res2x[-1]
         hi  = res2x[-1] + 3*abs(res2x[-1] - res2x[-2])
       else:
         low = res2x[-1] - 3*abs(res2x[-1] - res2x[-2])
         hi  = res2x[-1]
         print(low,hi)
       #res2x.sort()
       f = np.linspace(0.001,0.002,4)
       res2x+=f
       y = np.zeros((10,2))
       
       
       y[:,0] = np.linspace(low,hi,10)
       
       f = interpolate.interp1d(res2x, res2y, fill_value='extrapolate')
       y[:,1] = f(y[:,0])
       
       return y.astype(int)


def draw_predictions(frame, tracks):
    """Рисует предсказанные траектории"""
    predictor = TrajectoryPredictor()

    for (color_name, track_id), track in tracks.items():
        traj = track.get_trajectory()


        if len(traj) >= 5:
            d = traj[len(traj)-4:len(traj)]
            predictions = predict(d)
            if len(predictions) > 1:
                for i in range(1, len(predictions)):
                    alpha = i / len(predictions)
                    color = (int(255 * alpha), 0, int(255 * (1 - alpha)))
                    cv2.line(frame, (predictions[i - 1][0],predictions[i - 1][0]), (predictions[i][0],predictions[i][1]), color, 2, cv2.LINE_AA)

                # Последняя предсказанная точка
                if len(predictions)>0:
                    cv2.circle(frame, predictions[-1], 6, (0, 0, 255), 2)

    return frame


def draw_current_markers(frame, detected):
    """Рисует текущие обнаруженные маркеры"""
    for color_name, centers in detected.items():
        color_bgr = Config.HUE_RANGES[color_name]['bgr']
        for (x, y) in centers:
            cv2.circle(frame, (x, y), 10, color_bgr, 2)
            cv2.circle(frame, (x, y), 4, color_bgr, -1)
    return frame

def draw1(frame,a):
    if(len(a)==0):
        return frame
    a = sorted(a,key=lambda p: p[1])
    for i in range(0,len(a)-1):
        cv2.circle(frame, a[i], 8, (35,220,240), 3)
        cv2.line(frame,a[i-1],a[i],(35,220,240),3)
    cv2.circle(frame,a[0],8,(35,220,240),3)
    return frame

def draw_skeleton(frame, detected):
    """Рисует скелет по анатомии"""
    blue = detected['blue']  # левая рука
    green = detected['green']  # правая рука
    yellow = sorted(detected['yellow'],key=lambda p: p[0])  # голова, шея, тело, ноги
    draw = yellow

    neck = []
    
    leg1 = []
    leg2 = []
    print(draw)
    for i in range(0,min(3,len(draw))):
        leg1.append(draw[i])

    for i in range(3,min(5,len(draw))):
        neck.append(draw[i])
    
    for i in range(5,min(8,len(draw))):
        leg2.append(draw[i])
    
    frame = draw1(frame,neck)
    frame = draw1(frame,leg1)
    frame = draw1(frame,leg2)
    if len(blue)>0:
       blue = sorted(blue,key=lambda p: p[1])
       print(blue)
       for i in range(0,len(blue)-1):
          cv2.circle(frame, blue[i], 8, (255,0,0), 3)
          cv2.line(frame,blue[i-1],blue[i],(255,0,0),3)
       cv2.circle(frame,blue[0],8,(255,0,0),3)
    
    
        

    if len(green)>0:
       green = sorted(green,key=lambda p: p[1])
       for i in range(0,len(green)-1):
          cv2.circle(frame, green[i], 8, (0, 255, 0), 3)
          cv2.line(frame,green[i-1],green[i],(0,255,0),3)
       cv2.circle(frame,green[0],8,(0,255,0),3)
    return frame

   

  


# ============================================================
# 8. ЦВЕТОВЫЕ ПОЛОСЫ ДЛЯ ВИЗУАЛИЗАЦИИ
# ============================================================

def get_color_bar(h_min, h_max, color_bgr, height=80):
    """Создаёт цветовую полосу Hue"""
    bar = np.zeros((height, 360, 3), dtype=np.uint8)
    for x in range(360):
        hue = x // 2
        color = np.uint8([[[hue, 255, 255]]])
        bgr = cv2.cvtColor(color, cv2.COLOR_HSV2BGR)
        bar[:, x] = bgr[0, 0]

    cv2.rectangle(bar, (h_min * 2, 0), (h_max * 2, height), color_bgr, 3)
    cv2.putText(bar, f"Hue: {h_min}-{h_max}", (h_min * 2, height - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    return bar


def create_color_bars():
    """Создает цветовые полосы для всех цветов"""
    bars = []
    for color_name in Config.HUE_RANGES:
        h_min = cv2.getTrackbarPos(f'{color_name} H Min', 'Hue Settings')
        h_max = cv2.getTrackbarPos(f'{color_name} H Max', 'Hue Settings')
        bar = get_color_bar(h_min, h_max, Config.HUE_RANGES[color_name]['bgr'])
        cv2.putText(bar, Config.HUE_RANGES[color_name]['name'], (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        bars.append(bar)
    return np.vstack(bars)



def init_trackbars():
    cv2.namedWindow('Hue Settings')

    for color in Config.HUE_RANGES:
        cv2.createTrackbar(f'{color} H Min', 'Hue Settings', Config.HUE_RANGES[color]['min'], 180, lambda x: None)
        cv2.createTrackbar(f'{color} H Max', 'Hue Settings', Config.HUE_RANGES[color]['max'], 180, lambda x: None)

    cv2.createTrackbar('S Min', 'Hue Settings', Config.S_MIN, 255, lambda x: None)
    cv2.createTrackbar('S Max', 'Hue Settings', Config.S_MAX, 255, lambda x: None)
    cv2.createTrackbar('V Min', 'Hue Settings', Config.V_MIN, 255, lambda x: None)
    cv2.createTrackbar('V Max', 'Hue Settings', Config.V_MAX, 255, lambda x: None)

def main():
    print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    MARKER MOTION CAPTURE LITE - ПОЛНАЯ ВЕРСИЯ                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  🎨 ЦВЕТА МАРКЕРОВ:                                                          ║
║     🔵 СИНИЙ  - левая рука (3 маркера: плечо, локоть, кисть)                ║
║     🟢 ЗЕЛЁНЫЙ - правая рука (3 маркера: плечо, локоть, кисть)              ║
║     🟡 ЖЁЛТЫЙ  - голова, шея, бёдра, колени, стопы (всего 8)                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ✅ РЕАЛИЗОВАННЫЕ ТРЕБОВАНИЯ:                                                ║
║     1. Детекция маркеров 3 цветов                                           ║
║     2. Трекинг с историей 4 секунды                                         ║
║     3. Предсказание траектории (ML - линейная экстраполяция)                ║
║     4. Построение скелета                                                   ║
║     5. 5 обязательных CV картинок (нажмите D)                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  🎮 УПРАВЛЕНИЕ:                                                              ║
║     ESC - выход                                                             ║
║     D   - создать 5 обязательных CV картинок                                ║
║     S   - сохранить скриншот                                                ║
║     H   - показать справку                                                  ║
╚══════════════════════════════════════════════════════════════════════════════╝
    """)

    cap = cv2.VideoCapture('data/v.mp4')
    

    if not cap.isOpened():
        print("❌ ОШИБКА: Не удалось открыть камеру!")
        return

    init_trackbars()
    tracker = Tracker()

    # Пустое окно для Hue Settings
    dummy = np.zeros((200, 500, 3), dtype=np.uint8)
    cv2.imshow('Hue Settings', dummy)

    frame_count = 0
    fps = 0
    fps_start = cv2.getTickCount()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Детекция
        detected, masks = detect_all(hsv, None)

        # Трекинг
        tracks = tracker.update(detected)

        # Визуализация
        result = frame.copy()
       # result = draw_trajectories(result, tracks)  # История движения
        #result = draw_predictions(result, tracks)  # Предсказания
        result = draw_current_markers(result, detected)  # Текущие маркеры
        result = draw_skeleton(result, detected)  # Скелет

        # FPS
        if frame_count % 30 == 0:
            fps_end = cv2.getTickCount()
            fps = cv2.getTickFrequency() / (fps_end - fps_start)
            fps_start = fps_end

        # Информация
        info = f"FPS: {fps:.1f} | Blue: {len(detected['blue'])}/3 | Green: {len(detected['green'])}/3 | Yellow: {len(detected['yellow'])}/8"
        cv2.putText(result, info, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv2.putText(result, "D: CV images | S: Screenshot | ESC: Exit", (10, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

        # Цветовые полосы
        color_bars = create_color_bars()

        # Маски
        h, w = frame.shape[:2]
        mask_blue = cv2.cvtColor(masks['blue'], cv2.COLOR_GRAY2BGR)
        mask_green = cv2.cvtColor(masks['green'], cv2.COLOR_GRAY2BGR)
        mask_yellow = cv2.cvtColor(masks['yellow'], cv2.COLOR_GRAY2BGR)

        mask_blue[masks['blue'] > 0] = (255, 0, 0)
        mask_green[masks['green'] > 0] = (0, 255, 0)
        mask_yellow[masks['yellow'] > 0] = (0, 255, 255)

        masks_row1 = np.hstack([mask_blue, mask_green])
        masks_row2 = np.hstack([mask_yellow, np.zeros((h, w, 3), dtype=np.uint8)])
        masks_display = np.vstack([masks_row1, masks_row2])

        cv2.putText(masks_display, "BLUE (left arm)", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv2.putText(masks_display, "GREEN (right arm)", (w + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv2.putText(masks_display, "YELLOW (body/legs)", (10, h + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255),
                    2)

        # Показываем окна
        cv2.imshow('Skeleton & Tracking', cv2.resize(result,(600,600)))
        cv2.imshow('Color Bars',cv2.resize(color_bars,(600,600)))
        cv2.imshow('Masks', cv2.resize(masks_display,(600,600)))

        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC
            break
        
        elif key == ord('s'):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            cv2.imwrite(f'screenshot_{timestamp}.png', result)
            print(f"📸 Скриншот сохранен: screenshot_{timestamp}.png")
        elif key == ord('h'):
            print("""
=== СПРАВКА ===
ESC - выход
D   - создать 5 обязательных CV картинок
S   - сохранить скриншот
H   - показать справку

Настройка цветов:
- Двигайте ползунки H Min/Max для каждого цвета
- Смотрите на окно Masks - там белым видны маркеры
===============
""")

    cap.release()
    cv2.destroyAllWindows()
    print("\n✅ Программа завершена. Результаты сохранены в папке проекта.")


# ============================================================
# 12. ЗАПУСК
# ============================================================

if __name__ == "__main__":
    # Создаем папку для результатов
    
    main()


╔══════════════════════════════════════════════════════════════════════════════╗
║                    MARKER MOTION CAPTURE LITE - ПОЛНАЯ ВЕРСИЯ                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  🎨 ЦВЕТА МАРКЕРОВ:                                                          ║
║     🔵 СИНИЙ  - левая рука (3 маркера: плечо, локоть, кисть)                ║
║     🟢 ЗЕЛЁНЫЙ - правая рука (3 маркера: плечо, локоть, кисть)              ║
║     🟡 ЖЁЛТЫЙ  - голова, шея, бёдра, колени, стопы (всего 8)                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ✅ РЕАЛИЗОВАННЫЕ ТРЕБОВАНИЯ:                                                ║
║     1. Детекция маркеров 3 цветов                                           ║
║     2. Трекинг с историей 4 секунды                                         ║
║     3. Предсказание траектории (ML - линейная экстраполяция)                ║
║     4. Построение скелета           

In [ ]:
os.getcwd()

In [ ]:
import os
os.listdir('data')

In [ ]:
def make_video(save:str,model:YOLO):
 cap = cv2.VideoCapture(0)
 
 d = torch.zeros((1,10,3),device='cuda')
 while True:
    ret,frame = cap.read()
    res = model(frame)
    key = res[0].keypoints.data
    d = torch.concat((d,key[7:17]),dim=0)
    torch.save(d,save)
    cv2.imshow("fg",res[0].plot())
    if cv2.waitKey(1) & 0xFF == ord('q'):
       cv2.destroyAllWindows()
       del cap
       break
      

In [ ]:
make_video('videos/data.pt',model)

In [ ]:
def filter_color(color_to_filter):
    # Чтение изображения
    
    cap = cv2.VideoCapture(0)
    k = np.ones((5,5))
    while True:
     ret,frame = cap.read()
     frame = cv2.morphologyEx(frame, cv2.MORPH_OPEN, k)
     #frame = cv2.morphologyEx(frame, cv2.MORPH_CLOSE, k)
     fing = cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
     cv2.imwrite("filt.png",frame)
     #hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Определение диапазона цвета для фильтрации
     lower_bound = np.array([0,100,100]) 
     upper_bound = np.array([20,255,255])
     
    # Создание маски
     mask = cv2.inRange(fing, lower_bound, upper_bound)
    
    # Извлечение цвета из исходного изображения
     result = cv2.bitwise_and(fing, fing, mask=mask)
     cv2.imwrite("filt1.png",result)
     gr = cv2.cvtColor(result,cv2.COLOR_BGR2GRAY)
    
     #ed = cv2.Canny(gr,30,200)
      
     
     contours, hierarchy = cv2.findContours(gr,
                      cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
     for cont in contours:
       M = cv2.moments(cont)
       if M['m00']!=0:
         cx = int(M['m10']/M['m00'])
         cy = int(M['m01']/M['m00'])
    
       if cv2.contourArea(cont)>50:
         cv2.circle(frame, (cx, cy), 5, (0, 255,0), -1)
         cv2.drawContours(frame, [cont], -1, (0, 0,255), 3)


     cv2.imshow("fg",frame)
     if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        del cap
        break
    # Преобразование в HSV
     
filter_color(0)

In [ ]:

#отрисовка

def draw(model:YOLO):
  cap = cv2.VideoCapture(0)
  res1 = []
  while True:
    ret,frame = cap.read()
    res = model(frame)
    key = res[0].keypoints.data
    frame = res[0].plot(
    )
    for i in range(max(len(res1)-10,0),len(res1)):
     
     res2 = res1[i]
     for bbx in res2:
       for peop in bbx.keypoints.xy:
            p = np.array(peop.cpu().to(dtype = torch.int))
            cv2.polylines(frame,p.reshape(-1,1,2),True,color=(0,255,0),thickness=4)
    res1.append(res)
    cv2.imshow("fg",frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
       cv2.destroyAllWindows()
       del cap
       break

In [ ]:
data = torch.load('videos/data.pt')

In [ ]:
def dataset_make(path:str,window:int):
    data = torch.load(path)
    
    x = torch.zeros((1,window,2))
    y = torch.zeros((1,2))
    for i in range(0,data.shape[0]-window):
        for j in range(0,17):
            
            x = torch.concat((data[i:i+window,j,0:2].cpu().unsqueeze(0),x),dim=0)
            y = torch.concat((data[i+window,j,0:2].cpu().unsqueeze(0),y),dim=0)
    return (x,y)

In [ ]:
x,y = dataset_make('videos/data.pt',5)

In [ ]:
class net(torch.nn.Module):
    def __init__(self, input:int,output:int,hidden = 5):
        super().__init__()
        self.lstm = torch.nn.LSTM(input,hidden)
        self.linear = torch.nn.Linear(hidden,output)
    def forward(self,x):
        out,_ = self.lstm(x)
        out = out[:,-1,:]
        return self.linear(out)

class time_dataset(Dataset):
    def __init__(self, X, Y):

        self.x =X
        self.y  =Y
    def __getitem__(self, index):
        
        return (self.x[index], self.y[index])

    def __len__(self):
        
        return self.x.shape[0]

In [ ]:
import numpy as np
from scipy.interpolate import pchip_interpolate

In [ ]:
def draw2(model:YOLO,stdx:float,stdy:float,meanx:float,meany:float):
  cap = cv2.VideoCapture(0)
  d = []
  while True:
    ret,frame = cap.read()
    res = model1(frame)
    key = res[0].keypoints.data
    
    
    frame = res[0].plot(
    )
    if len(d)>5:
     res2 = np.array(d[len(d)-4:len(d)])
     
     x_new = np.linspace([res[-1,0] + abs(res[-2,0]-res[-1,0])])
     y_new = pchip_interpolate(res2[:,0], res[:,1], x_new)
     cv2.circle(int(x_new),int(y_new[0]),7,thickness=5)
    
    
    cv2.imshow("fg",frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
       cv2.destroyAllWindows()
       del cap
       break

In [ ]:
from sklearn.model_selection import train_test_split

x = np.array(x)
y = np.array(y)
x_tr,x_val,y_tr,y_val = train_test_split(x,y,test_size=0.3)
stdx = x_tr[:,:,0].std()
stdy = x_tr[:,:,1].std()
meanx = x_tr[:,:,0].mean()
meany = x_tr[:,:,1].mean()

stdxy = y_tr[:,0].std()
stdyy = y_tr[:,1].std()
meanxy= y_tr[:,0].mean()
meanyy = y_tr[:,1].mean()
x_tr[:,:,0] = (x_tr[:,:,0] - meanx)/stdx
x_tr[:,:,1] = (x_tr[:,:,1] - meany)/stdy
y_tr[:,0] = (y_tr[:,0]-meanxy)/stdxy
y_tr[:,1] = (y_tr[:,1]-meanyy)/stdyy
x_tr = torch.tensor(x_tr)
y_tr = torch.tensor(y_tr)


x_val[:,:,0] = (x_val[:,:,0] - meanx)/stdx
x_val[:,:,1] = (x_val[:,:,1] - meany)/stdy
y_val[:,0] = (y_val[:,0]-meanxy)/stdxy
y_val[:,1] = (y_val[:,1]-meanyy)/stdyy
x_val = torch.tensor(x_val)
y_val = torch.tensor(y_val)


dataset = time_dataset(x_tr,y_tr)
dataset_val = time_dataset(x_val,y_val)
dataload = DataLoader(dataset,batch_size=128)
dataload_val = DataLoader(dataset_val,batch_size=128)
model = net(2,2,10)
model = model.to(device=device)
lost1 = {'loss':[],'val':[]}

loss_f = torch.nn.MSELoss()
optim = torch.optim.Adam(
    model.parameters(),
    lr=3e-4
)
epoch = 1000
for i in range(0,epoch):
    losttr = 0.0
    lost_val = 0.0
    col = 0
    col1 = 0
    for x_batch,y_batch in dataload:
        optim.zero_grad()
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        y_pred = model(x_batch)
        lost = loss_f(y_pred,y_batch)
        lost.backward()
        optim.step()
        losttr+=lost.item()
        col+=1
    for x_batch,y_batch in dataload_val:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        y_pred = model(x_batch)
        lost = loss_f(y_pred,y_batch)
        lost.backward()
        lost_val+=lost.item()
        col1+=1
    lost1['loss'].append(losttr/col)
    lost1['val'].append(lost_val/col1)

    


    clear_output(True)
    fig,ax = plt.subplots(1,figsize = (12,8))
    plt.title('train_los')
    plt.grid()
    plt.plot(lost1['loss'],'o-',label = 'trainer')
    plt.plot(lost1['val'],'o-',label = 'val')
    plt.legend()
    plt.show()



In [ ]:
from scipy import interpolate
def draw2(model1:YOLO):
  cap = cv2.VideoCapture(0)
  d = torch.zeros((1,2))
  while True:
    ret,frame = cap.read()
    res = model1(frame)
    if(len(res)>0):
      if len(res[0].keypoints.data)>0:
         key = res[0].keypoints.data[0].cpu()
      else:
         continue
    
   
     
     
      frame = res[0].plot()
      if len(d)>4:
      
       res2 = d[len(d)-4:len(d)]
       res2 = torch.tensor(res2)
       
       
       last = res2[-1,0]
       pr = res2[-2,0]
       res2 = np.array(sorted(res2.cpu().detach().tolist(),key=lambda p: p[0]))
       
       res2x= res2[:,0:1].reshape(4)
       res2y = res2[:,1:2].reshape(4)
       
       
       low = 0
       hi = 0
       if last>pr:
         low = res2x[-1]
         hi  = res2x[-1] + 3*abs(res2x[-1] - res2x[-2])
       else:
         low = res2x[-1] - 3*abs(res2x[-1] - res2x[-2])
         hi  = res2x[-1]
         print(low,hi)
       #res2x.sort()
       f = np.linspace(0.001,0.002,4)
       res2x+=f
       y = np.zeros((10,2))
       
       
       y[:,0] = np.linspace(low,hi,10)
       
       f = interpolate.interp1d(res2x, res2y, fill_value='extrapolate')
       y[:,1] = f(y[:,0])
       
      
       cv2.polylines(frame,y.reshape((10,1,2)).astype(int),True,color=(0,255,0),thickness=6)
       #cv2.circle(frame,center=(int(y[0][0]),int(y[0][1])),radius=5,thickness=10,color = (255,0,0))
      d = torch.concat((d,key[10][0:2].unsqueeze(0)),dim=0)
    
    
    cv2.imshow("fg",frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
       cv2.destroyAllWindows()
       del cap
       break
draw2(model1)

In [ ]:
cap = cv2.VideoCapture(0)
del cap

In [ ]:
del cap

In [ ]:
x = torch.tensor([[3, 1, 4], [1, 5, 9], [2, 6, 5]])
sorted_values, sorted_indices = torch.sort(x, dim=0)
print("Отсортированные значения:", sorted_values)
print("Индексы:", sorted_indices)

In [ ]:
lost.item()